In [149]:
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import os
import random
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
def extract_mediapipe_to_csv(data_path: str, save_path: str, model_path: str):

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]
    static = not any(data_path.lower().endswith(ext) for ext in video_extensions)

    base_options = python.BaseOptions(model_asset_path=model_path)

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,  
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    data = []

    if static:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            image = cv2.imread(data_path)
            if image is None:
                raise ValueError(f"Could not read image: {data_path}")

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            results = landmarker.detect(mp_image)

            if results.pose_landmarks:
                row = {"FrameNo": 0}

                for joint in JOINT_ORDER:
                    idx = LANDMARK_INDEX[joint]
                    lm = results.pose_landmarks[0][idx]

                    row[f"{joint}_x"] = lm.x
                    row[f"{joint}_y"] = lm.y
                    row[f"{joint}_z"] = lm.z

                data.append(row)

    else:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            cap = cv2.VideoCapture(data_path)

            if not cap.isOpened():
                raise ValueError(f"Could not open video: {data_path}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                timestamp_ms = int((frame_idx / fps) * 1000)
                results = landmarker.detect_for_video(mp_image, timestamp_ms)

                if results.pose_landmarks:
                    row = {"FrameNo": frame_idx}

                    for joint in JOINT_ORDER:
                        idx = LANDMARK_INDEX[joint]
                        lm = results.pose_landmarks[0][idx]

                        row[f"{joint}_x"] = lm.x
                        row[f"{joint}_y"] = lm.y
                        row[f"{joint}_z"] = lm.z

                    data.append(row)

                frame_idx += 1

            cap.release()

    # ---- SAVE CSV ----
    columns = ["FrameNo"]
    for joint in JOINT_ORDER:
        columns += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    df = pd.DataFrame(data)
    df = df[columns]

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)

    print(f"Saved {len(df)} rows to {save_path}")

In [ ]:
""""
video_folder = "../data"
model_path = "../data/pose_landmarker.task"
output_folder = "../Jakob_movement_CSV"

video_extensions = [".mp4", ".mov", ".avi", ".mkv"]

for file_name in os.listdir(video_folder):
    
    if not any(file_name.lower().endswith(ext) for ext in video_extensions):
        continue

    video_path = os.path.join(video_folder, file_name)

    base_name = os.path.splitext(file_name)[0]
    save_name = f"{base_name}_mediapipe.csv"
    save_path = os.path.join(output_folder, save_name)

    print(f"Processing: {file_name}")

    extract_mediapipe_to_csv(
        data_path=video_path,
        save_path=save_path,
        model_path=model_path
    )
print("Done.")
"""

Processing: Upphöjd-still.mov


I0000 00:00:1776667726.393365 16613759 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776667726.449128 16613763 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776667726.465099 16613762 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776667726.546367 16613763 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Saved 132 rows to ../Jakob_movement_CSV/Upphöjd-still_mediapipe.csv
Processing: mark-rörelse.mov


I0000 00:00:1776667728.192471 16613810 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1776667728.277043 16613814 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776667728.287273 16613812 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 271 rows to ../Jakob_movement_CSV/mark-rörelse_mediapipe.csv
Processing: upphöjd-rörelse.mov


I0000 00:00:1776667731.483819 16613857 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1776667731.541748 16613861 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776667731.549318 16613865 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 360 rows to ../Jakob_movement_CSV/upphöjd-rörelse_mediapipe.csv
Processing: mark-still.mov


I0000 00:00:1776667736.003474 16613949 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1776667736.051647 16613952 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776667736.058139 16613956 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 119 rows to ../Jakob_movement_CSV/mark-still_mediapipe.csv
Done.


## Compare mediapipe with irl measurments

In [77]:
# Scale factors
df = pd.read_csv("../Jakob_movement_CSV/mark-still_mediapipe.csv")
# Ground truth
ground_truth = {
    "hip_to_shoulder":      50,
    "knee_to_hip":          44,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}

x_scale = ground_truth["hip_to_hip"] / (df["left_hip_x"] - df["right_hip_x"]).abs().mean()
y_scale = ground_truth["hip_to_shoulder"] / (df["left_hip_y"] - df["left_shoulder_y"]).abs().mean()

print(f"X scale: {x_scale:.2f} cm per unit")
print(f"Y scale: {y_scale:.2f} cm per unit")

# Scale all coordinates
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * x_scale
    df[f"{joint}_y"] = df[f"{joint}_y"] * y_scale

# Now calculate Euclidean in cm
def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

df["hip_to_shoulder"]      = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]           = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]        = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"] = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                        df["right_shoulder_x"], df["right_shoulder_y"])
df["shoulder_to_shoulder"] = np.mean(df["left_shoulder_x"] - df["right_shoulder_x"])

avg = df[["hip_to_shoulder", "knee_to_hip", "hip_to_hip", 
          "knee_to_ankle", "shoulder_to_shoulder"]].mean()

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key]
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")

X scale: 468.87 cm per unit
Y scale: 238.31 cm per unit
Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 50.0            50.7          0.7        1.5%
knee_to_hip                     44.0            41.1          2.9        6.6%
hip_to_hip                      25.0            25.0          0.0        0.0%
knee_to_ankle                   43.0            37.2          5.8       13.4%
shoulder_to_shoulder            38.0            40.8          2.8        7.3%


In [128]:
# Scale factors
df = pd.read_csv("../Jakob_movement_CSV/upphöjd-still_mediapipe.csv")
# Ground truth
ground_truth = {
    "hip_to_shoulder":      50,
    "knee_to_hip":          44,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}

x_scale = ground_truth["hip_to_hip"] / (df["left_hip_x"] - df["right_hip_x"]).abs().mean()
y_scale = ground_truth["hip_to_shoulder"] / (df["left_hip_y"] - df["left_shoulder_y"]).abs().mean()

print(f"X scale: {x_scale:.2f} cm per unit")
print(f"Y scale: {y_scale:.2f} cm per unit")

# Scale all coordinates
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * x_scale
    df[f"{joint}_y"] = df[f"{joint}_y"] * y_scale

# Now calculate Euclidean in cm
def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

df["hip_to_shoulder"]      = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]           = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]        = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"] = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                        df["right_shoulder_x"], df["right_shoulder_y"])

avg = df[["hip_to_shoulder", "knee_to_hip", "hip_to_hip", 
          "knee_to_ankle", "shoulder_to_shoulder"]].mean()

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key]
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")

X scale: 483.64 cm per unit
Y scale: 241.97 cm per unit
Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 50.0            50.7          0.7        1.4%
knee_to_hip                     44.0            39.7          4.3        9.7%
hip_to_hip                      25.0            25.0          0.0        0.0%
knee_to_ankle                   43.0            35.5          7.5       17.5%
shoulder_to_shoulder            38.0            42.4          4.4       11.5%


In [148]:
import numpy as np

# Load your mediapipe CSV

df = pd.read_csv("../Jakob_movement_CSV/upphöjd-still_mediapipe.csv")


FRAME_WIDTH = 1620  
FRAME_HEIGHT = 1080  

# Convert normalized coords to pixels
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * FRAME_WIDTH
    df[f"{joint}_y"] = df[f"{joint}_y"] * FRAME_HEIGHT


def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

# Calculate distances per frame
df["hip_to_shoulder"]        = euclidean(df["left_hip_x"], df["left_hip_y"],
                                    df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]            = euclidean(df["left_knee_x"], df["left_knee_y"],
                                    df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]             = euclidean(df["left_hip_x"], df["left_hip_y"],
                                    df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                    df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"]   = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                    df["right_shoulder_x"], df["right_shoulder_y"])

# Average across frames
avg = df[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip",
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].mean()

std_devs = df[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip", 
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].std()

print("\nConsistency Check (lower std = more stable):")
for key in ground_truth.keys():
    print(f"{key}: ±{std_devs[key]:.2f} cm")

if std_devs.mean() > 2.0:
    print("\n⚠️ Warning: High variation detected. Person may have moved during recording.")
    
# Ground truth
ground_truth = {
    "hip_to_shoulder":      50,
    "knee_to_hip":          44,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}


# Scale using hip_to_shoulder as reference
scale = ground_truth["hip_to_shoulder"] / avg["hip_to_shoulder"]
print(f"Scale factor: {scale:.2f} cm per unit\n")

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key] * scale
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")


Consistency Check (lower std = more stable):
hip_to_shoulder: ±0.90 cm
knee_to_hip: ±0.61 cm
hip_to_hip: ±0.47 cm
knee_to_ankle: ±1.11 cm
shoulder_to_shoulder: ±0.47 cm
Scale factor: 0.22 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 50.0            50.0          0.0        0.0%
knee_to_hip                     44.0            39.4          4.6       10.5%
hip_to_hip                      25.0            18.6          6.4       25.6%
knee_to_ankle                   43.0            35.2          7.8       18.2%
shoulder_to_shoulder            38.0            31.5          6.5       17.0%


## Data augmentation 

* mirror the movement
* rotate it around y dimension
* add some extra noise to x, y and z

In [ ]:
def mirror_data(df, center = 0):
    df_flipped = df.copy()

    joint_pairs = [
        ('left_shoulder', 'right_shoulder'),
        ('left_elbow', 'right_elbow'),
        ('left_hand', 'right_hand'),
        ('left_hip', 'right_hip'),
        ('left_knee', 'right_knee'),
        ('left_foot', 'right_foot')
    ]

    for left_joint, right_joint in joint_pairs:
        # Swap + mirror x
        df_flipped[f"{left_joint}_x"] = center - df[f"{right_joint}_x"]
        df_flipped[f"{right_joint}_x"] = center - df[f"{left_joint}_x"]

        # Swap y
        df_flipped[f"{left_joint}_y"] = df[f"{right_joint}_y"]
        df_flipped[f"{right_joint}_y"] = df[f"{left_joint}_y"]

        # Z: swap 
        df_flipped[f"{left_joint}_z"] = df[f"{right_joint}_z"]
        df_flipped[f"{right_joint}_z"] = df[f"{left_joint}_z"]

    # Head (no swap, only mirror)
    df_flipped["head_x"] = center - df["head_x"]
    df_flipped["head_y"] = df["head_y"]
    df_flipped["head_z"] = df["head_z"]

    return df_flipped


def rotate_y(df, angle_deg):
    theta = np.radians(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    df_rot = df.copy()

    for col in df.columns:
        if col.endswith("_x"):
            joint = col[:-2]
            x = df[f"{joint}_x"]
            z = df[f"{joint}_z"]

            df_rot[f"{joint}_x"] = x * cos_t + z * sin_t
            df_rot[f"{joint}_z"] = -x * sin_t + z * cos_t

    return df_rot


def add_noise_to_pose(df, noise_std_xy=0.002, noise_std_z=0.001, 
                      apply_to_z=True, seed=None):
    
    if seed is not None:
        np.random.seed(seed)
    
    df_noisy = df.copy()
    
    joints = ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]
    
    for joint in joints:
        # Add noise to X and Y 
        df_noisy[f"{joint}_x"] += np.random.normal(0, noise_std_xy, len(df))
        df_noisy[f"{joint}_y"] += np.random.normal(0, noise_std_xy, len(df))
        
        # Add noise to Z (optional, smaller magnitude)
        if apply_to_z:
            df_noisy[f"{joint}_z"] += np.random.normal(0, noise_std_z, len(df))
    
    return df_noisy